In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

In [16]:
dataset_train = pd.read_csv("train.csv")
dataset_test = pd.read_csv("test.csv")

In [17]:
binary_cols = ["internet_access"]

ordinal_cols = ["sleep_quality", "facility_rating", "exam_difficulty"]

ordinal_categories = [
    ["poor", "average", "good"],
    ["low", "medium", "high"],
    ["easy", "moderate", "hard"]
]

onehot_cols = ["gender", "course", "study_method"]

num_cols = ["age", "study_hours", "class_attendance", "sleep_hours"]

In [ ]:
# BINARY ENCODING 
binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["no", "yes"]]))
])

# ORDINARY ENCODING
ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=ordinal_categories))
])

# ONE-HOT ENCODING
onehot_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# NUMERIC SCALING
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# PREPROCESSING
preprocessor = ColumnTransformer(
    transformers=[
        ("bin", binary_transformer, binary_cols),
        ("ord", ordinal_transformer, ordinal_cols),
        ("oh", onehot_transformer, onehot_cols),
        ("num", numeric_transformer, num_cols)
    ],
    remainder="drop"
)

#  XGB
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", xgb_model)
])

# LGB
lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", lgb_model)
])


In [19]:
X = dataset_train.drop(columns=["id", "exam_score"])
y = dataset_train["exam_score"]

X_test = dataset_test.drop(columns=["id"])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [20]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgb_pipeline.fit(X_train, y_train)

    # OOF predictions
    oof_preds[val_idx] = lgb_pipeline.predict(X_val)

    # test predictions (averaged later)
    test_preds += lgb_pipeline.predict(X_test) / kf.n_splits

In [21]:
rmse = root_mean_squared_error(y, oof_preds)
print("OOF RMSE:", rmse)

OOF RMSE: 8.756784347866576
